In [ ]:
# 1. 필수 라이브러리 설치 및 임포트 (최신버전 유지 권장)
!pip install -U ultralytics roboflow

import time
import pandas as pd
from ultralytics import YOLO
from roboflow import Roboflow
from IPython.display import Image, display

# 2. 데이터셋 다운로드 (버전 2 및 battery 반영)
rf = Roboflow(api_key="kOXhWfzN0JyIGBAuZ0Ap")
project = rf.workspace("seunghopark").project("bomb_wire_v2")
version = project.version(2)
dataset = version.download("yolov8")

# 3. 모델 학습 설정 (YOLO26n 적용)
# 👉 핵심 포인트: 최신판인 yolo26n.pt를 호출합니다!
model = YOLO('yolo26n.pt')

# 결과 저장 폴더 이름을 YOLO26 모델에 맞게 지정
project_name = 'bomb_wire_battery_v2_study_yolo26n'

print("\n" + "="*50)
print(f"🚀 최신 SOTA 모델 YOLO26n Early Stopping 모드로 학습 시작... (저장 폴더: {project_name})")
print("="*50)

start_time = time.time()

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=500,           # 최대 학습 횟수
    patience=50,          # 50 epoch 동안 개선 없으면 조기 종료
    imgsz=640,
    plots=True,
    name=project_name     # 새로운 결과 폴더명 적용
)

end_time = time.time()
training_duration = end_time - start_time

# 4. 학습 시간 및 F1 Score 출력
print("\n" + "="*50)
print(f"✅ 학습 종료! 총 소요 시간: {training_duration:.2f}초 ({training_duration/60:.2f}분)")

# 검증 세트를 통해 F1 Score 등 최종 메트릭 확인
metrics = model.val()
f1_score = metrics.box.f1.mean()
print(f"⭐ 최종 F1 Score (Mean): {f1_score:.4f}")
print("="*50)

# 5. 시각화: 학습 결과 차트 (results.png)
print("\n📊 학습 과정 그래프:")
result_plot_path = f"runs/detect/{project_name}/results.png"
display(Image(filename=result_plot_path, width=1000))

# 6. CSV 결과 파일 확인
print("\n📝 CSV 로그 데이터 (하단 5행):")
csv_path = f"runs/detect/{project_name}/results.csv"
df = pd.read_csv(csv_path)
display(df.tail())
print(f"\n📂 CSV 파일 경로: {csv_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 149.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings wit


Extracting Dataset Version Zip to bomb_wire_v2-2 in yolov8:: 100%|██████████| 960/960 [00:00<00:00, 9658.29it/s]



🚀 최신 SOTA 모델 YOLO26n Early Stopping 모드로 학습 시작... (저장 폴더: bomb_wire_battery_v2_study_yolo26n)
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/bomb_wire_v2-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bomb_wire_batt

In [ ]:
import glob
import cv2
import random
from ultralytics import YOLO
from google.colab.patches import cv2_imshow # 코랩 전용 이미지 출력 함수

# 1. 아까 학습한 최고 성능의 모델(best.pt) 불러오기
project_name = 'bomb_wire_battery_v2_study_model'
model_path = f"/content/runs/detect/bomb_wire_battery_v2_study_yolo26n/weights/best.pt"
best_model = YOLO(model_path)

print("🚀 테스트 이미지에 바운딩 박스를 그리기 시작합니다...")

# 2. 테스트할 이미지 경로 설정 (Roboflow 다운로드 폴더 내의 test/images)
test_images_path = f"{dataset.location}/test/images"

# 3. 예측(Predict) 실행
# conf=0.5 옵션은 "확률이 50% 이상인 객체만 박스를 쳐라" 라는 뜻입니다.
results = best_model.predict(source=test_images_path, save=True, conf=0.5)

# 4. 예측 결과가 저장된 폴더 경로 자동 추적
predict_dir = results[0].save_dir
image_paths = glob.glob(f"{predict_dir}/*.jpg")

# 5. 결과 시각화 (스크린샷 방식 반영)
print("\n" + "="*50)
print(f"📂 바운딩 박스가 그려진 이미지 저장 위치: {predict_dir}")
print("="*50)

if not image_paths:
    print("❌ 이미지 파일이 없습니다. 경로를 확인해주세요.")
else:
    # 이미지가 너무 많으면 화면이 멈출 수 있으므로, 랜덤하게 3장만 뽑아서 보여줍니다.
    print(f"총 {len(image_paths)}장의 테스트 이미지 중 3장을 랜덤으로 보여줍니다:\n")

    selected_images = random.sample(image_paths, min(3, len(image_paths)))

    for image_path in selected_images:
        img = cv2.imread(image_path)
        cv2_imshow(img) # 코랩 화면에 이미지 띄우기

In [ ]:
import pandas as pd
import os
import numpy as np

# 1. 파일 경로 설정
project_name = 'bomb_wire_battery_v2_study_model'
csv_path = f"/content/runs/detect/bomb_wire_battery_v2_study_yolo26n/results.csv"

# 기존 CSV 읽기
df_orig = pd.read_csv(csv_path)

# 2. [오류 복구] 이전 잘못된 실행으로 생긴 중복/쓰레기 데이터 청소
# 2-1. 문자열 요약행 제거 (epoch 열이 숫자인 원래 데이터만 남김)
cols = df_orig.columns
col_epoch = [c for c in cols if 'epoch' in c][0]
df_orig = df_orig[pd.to_numeric(df_orig[col_epoch], errors='coerce').notnull()]

# 2-2. 잘못 만들어진 이름의 열(컬럼)이 있다면 삭제
wrong_cols = [c for c in df_orig.columns if c in ['지연시간(ms)', '학습시간(s)']]
if wrong_cols:
    df_orig = df_orig.drop(columns=wrong_cols)

# 컬럼 리스트 갱신
cols = df_orig.columns
col_p = [c for c in cols if 'precision' in c][0]
col_r = [c for c in cols if 'recall' in c][0]
col_map50 = [c for c in cols if 'mAP50(B)' in c][0]
col_map = [c for c in cols if 'mAP50-95' in c][0]

# 3. 기본 정확도 데이터 준비
class_names = list(metrics.names.values())
precisions = metrics.box.p.tolist()
recalls = metrics.box.r.tolist()
f1_scores = metrics.box.f1.tolist()
ap50_list = metrics.box.ap50.tolist()
ap_list = metrics.box.ap.tolist()

# 4. 속도, FPS 계산
if hasattr(metrics, 'speed'):
    speed = metrics.speed
    preprocess = round(speed.get('preprocess', 0), 2)
    inference = round(speed.get('inference', 0), 2)
    postprocess = round(speed.get('postprocess', 0), 2)

    total_latency = round(preprocess + inference + postprocess, 2)
    fps = round(1000 / total_latency, 2) if total_latency > 0 else '-'
else:
    total_latency = fps = '-'

# 5. 모델 복잡도 추출
try:
    total_params = sum(p.numel() for p in model.model.parameters())
    params_m = round(total_params / 1_000_000, 2)

    from ultralytics.utils.torch_utils import get_flops
    gflops = round(get_flops(model.model), 2)
except Exception as e:
    params_m = "-"
    gflops = "-"

# 6. 총 학습시간 계산
time_col = [c for c in cols if c.strip() == 'time']

if 'training_duration' in globals() and isinstance(training_duration, (int, float)):
    final_time = round(training_duration, 2)
elif time_col:
    time_series = pd.to_numeric(df_orig[time_col[0]], errors='coerce')
    calculated_time = round(time_series.sum(), 2)
    final_time = calculated_time if calculated_time > 0 else '-'
else:
    final_time = '-'

additional_rows = []

# 7. 클래스별 행 생성 (여기서도 올바른 이름표 사용)
for i, name in enumerate(class_names):
    row = {col: '' for col in cols}
    row[col_epoch] = f"Class: {name}"
    row[col_p] = round(precisions[i], 4)
    row[col_r] = round(recalls[i], 4)
    row[col_map50] = round(ap50_list[i], 4)
    row[col_map] = round(ap_list[i], 4)
    row['F1-Score'] = round(f1_scores[i], 4)

    # 🚨 이름표 확실하게 매칭
    row['추론속도(ms)'] = '-'
    row['총학습시간(s)'] = '-'
    row['FPS'] = '-'
    row['Params'] = '-'
    row['GFLOPs'] = '-'
    additional_rows.append(row)

# 8. 마지막 합계(ALL) 행 생성
summary_row = {col: '' for col in cols}
summary_row[col_epoch] = "SUMMARY (ALL)"
summary_row[col_p] = round(metrics.box.mp, 4)
summary_row[col_r] = round(metrics.box.mr, 4)
summary_row[col_map50] = round(metrics.box.map50, 4)
summary_row[col_map] = round(metrics.box.map, 4)
summary_row['F1-Score'] = round(metrics.box.f1.mean(), 4)

# 🚨 결과값 올바른 자리에 삽입
summary_row['추론속도(ms)'] = total_latency
summary_row['총학습시간(s)'] = final_time
summary_row['FPS'] = fps
summary_row['Params'] = params_m
summary_row['GFLOPs'] = gflops

additional_rows.append(summary_row)

# 9. 덮어쓰기 및 저장
df_additional = pd.DataFrame(additional_rows)
# 만약 기존 df_orig에 없는 새 컬럼('FPS' 등)이 additional_rows에 있다면 자동 추가됨
df_final = pd.concat([df_orig, df_additional], ignore_index=True)

# 컬럼 순서 정렬 (보기 편하게)
cols_order = [c for c in df_final.columns if c not in ['F1-Score', '추론속도(ms)', '총학습시간(s)', 'FPS', 'Params', 'GFLOPs']]
cols_order += ['F1-Score', '추론속도(ms)', '총학습시간(s)', 'FPS', 'Params', 'GFLOPs']
df_final = df_final[cols_order]

df_final.to_csv(csv_path, index=False, encoding='utf-8-sig')

print("\n✅ 누락된 항목 없이 모든 지표가 제자리에 업데이트되었습니다!")
display(df_final.tail(len(class_names) + 1))